In [1]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
#from scvi.external import MRVI
from mrvi_torch import TorchMRVI as MRVI

scvi.settings.seed = 0  # optional: ensures reproducibility
print("Last run with scvi-tools version:", scvi.__version__)
#save_dir = tempfile.TemporaryDirectory()
save_dir = '~/mrvi-reproducibility/tmp'
adata_path = os.path.join(save_dir, "haniffa_tutorial_subset.h5ad")

adata = sc.read(adata_path, backup_url="https://figshare.com/ndownloader/files/46017615")
sc.pp.highly_variable_genes(
    adata, n_top_genes=10000, inplace=True, subset=True, flavor="seurat_v3"
)
sample_key = "patient_id"  # target covariate
# batch_key="Site"  # nuisance variable identifier
#MRVI.setup_anndata(adata, sample_key=sample_key, backend="torch")
MRVI.setup_anndata(adata, sample_key=sample_key)


from sklearn.metrics import silhouette_score, silhouette_samples

/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[rank: 0] Seed set to 0


Last run with scvi-tools version: 1.4.0.post1


In [5]:
import os
import re

# Assume MRVI and adata already exist.

models_dir = "models"
pattern = re.compile(r"mrvi_model-(\d+)_(\d+)_(\d+)_(\d+)$")

# Dictionary to store final validation ELBOs
# Key: (a, b, c, d), Value: final validation ELBO
final_elbo_dict = {}
silhouette_score_dict = {}

for name in os.listdir(models_dir):
    full_path = os.path.join(models_dir, name)

    # Only consider directories
    if not os.path.isdir(full_path):
        continue

    # Match names of the form mrvi_model-a_b_c_d
    match = pattern.fullmatch(name)
    if match is None:
        continue

    a, b, c, d = map(int, match.groups())

    # Load the model
    model = MRVI.load(full_path, adata=adata)

    # Get the final validation ELBO
    final_elbo = list(model.history["elbo_validation"].iloc[-1])[0]

    # Store in dictionary
    final_elbo_dict[(a, b, c, d)] = final_elbo
    
    u = model.get_latent_representation()
    adata.obsm["u"] = u
    X = u  # or "z", or a UMAP if you prefer

    # labels: which cluster each cell belongs to
    labels = adata.obs["initial_clustering"].values  # or another column with cell-type annotations

    # Global silhouette score
    sil_global = silhouette_score(X, labels)
    print("Global silhouette score (u vs initial_clustering):", sil_global)

    # Per-cell silhouette scores
    sil_per_cell = silhouette_samples(X, labels)

    # Store in adata if you like
    adata.obs["silhouette_initial_clustering"] = sil_per_cell


INFO     File models/mrvi_model-2_1_32_1/model.pt already downloaded                                               
# DecoderZX Initialized with 2 hidden and 1 layers
# EncoderUZ Initialized with 32 hidden and 1 layers
INFO     File models/mrvi_model-2_2_32_1/model.pt already downloaded                                               


/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...


# DecoderZX Initialized with 2 hidden and 2 layers
# EncoderUZ Initialized with 32 hidden and 1 layers
INFO     File models/mrvi_model-2_3_32_1/model.pt already downloaded                                               
# DecoderZX Initialized with 2 hidden and 3 layers
# EncoderUZ Initialized with 32 hidden and 1 layers
INFO     File models/mrvi_model-4_1_32_1/model.pt already downloaded                                               
# DecoderZX Initialized with 4 hidden and 1 layers
# EncoderUZ Initialized with 32 hidden and 1 layers
INFO     File models/mrvi_model-4_2_32_1/model.pt already downloaded                                               


/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyf

# DecoderZX Initialized with 4 hidden and 2 layers
# EncoderUZ Initialized with 32 hidden and 1 layers
INFO     File models/mrvi_model-4_3_32_1/model.pt already downloaded                                               
# DecoderZX Initialized with 4 hidden and 3 layers
# EncoderUZ Initialized with 32 hidden and 1 layers


...

/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyf

# DecoderZX Initialized with 32 hidden and 1 layers
# EncoderUZ Initialized with 64 hidden and 1 layers
INFO     File models/mrvi_model-32_1_64_2/model.pt already downloaded                                              
# DecoderZX Initialized with 32 hidden and 1 layers
# EncoderUZ Initialized with 64 hidden and 2 layers
INFO     File models/mrvi_model-32_1_64_3/model.pt already downloaded                                              
# DecoderZX Initialized with 32 hidden and 1 layers
# EncoderUZ Initialized with 64 hidden and 3 layers
INFO     File models/mrvi_model-32_1_128_1/model.pt already downloaded                                             


/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyf

# DecoderZX Initialized with 32 hidden and 1 layers
# EncoderUZ Initialized with 128 hidden and 1 layers
INFO     File models/mrvi_model-32_1_128_2/model.pt already downloaded                                             
# DecoderZX Initialized with 32 hidden and 1 layers
# EncoderUZ Initialized with 128 hidden and 2 layers
INFO     File models/mrvi_model-32_1_128_3/model.pt already downloaded                                             
# DecoderZX Initialized with 32 hidden and 1 layers
# EncoderUZ Initialized with 128 hidden and 3 layers


/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...
/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...


In [7]:
# Print final validation ELBO sorted by value (ascending)
for (a, b, c, d), elbo in sorted(final_elbo_dict.items(), key=lambda x: x[1]):
    print(f"(a={a}, b={b}, c={c}, d={d}) -> final validation ELBO: {elbo}")


(a=8, b=2, c=32, d=1) -> final validation ELBO: 1411.47998046875
(a=2, b=3, c=32, d=1) -> final validation ELBO: 1411.6121826171875
(a=2, b=1, c=32, d=1) -> final validation ELBO: 1411.61669921875
(a=32, b=1, c=64, d=2) -> final validation ELBO: 1411.6856689453125
(a=4, b=3, c=32, d=1) -> final validation ELBO: 1411.7008056640625
(a=8, b=1, c=32, d=1) -> final validation ELBO: 1411.74560546875
(a=4, b=2, c=32, d=1) -> final validation ELBO: 1411.7562255859375
(a=32, b=1, c=64, d=3) -> final validation ELBO: 1411.7593994140625
(a=2, b=2, c=32, d=1) -> final validation ELBO: 1411.785400390625
(a=32, b=1, c=64, d=1) -> final validation ELBO: 1411.82666015625
(a=32, b=1, c=128, d=2) -> final validation ELBO: 1411.859375
(a=16, b=3, c=32, d=1) -> final validation ELBO: 1411.8804931640625
(a=32, b=1, c=128, d=1) -> final validation ELBO: 1411.890380859375
(a=16, b=2, c=32, d=1) -> final validation ELBO: 1411.9674072265625
(a=128, b=2, c=32, d=1) -> final validation ELBO: 1411.996826171875
(a

In [11]:
import numpy as np
from sklearn.metrics import silhouette_score, silhouette_samples

# X: MRVI latent representation (e.g. "u")
model = MRVI.load("models/mrvi_model-2_1_32_1", adata=adata)
u = model.get_latent_representation()
adata.obsm["u"] = u
X = u  # or "z", or a UMAP if you prefer

# labels: which cluster each cell belongs to
labels = adata.obs["initial_clustering"].values  # or another column with cell-type annotations

# Global silhouette score
sil_global = silhouette_score(X, labels)
print("Global silhouette score (u vs initial_clustering):", sil_global)

# Per-cell silhouette scores
sil_per_cell = silhouette_samples(X, labels)

# Store in adata if you like
adata.obs["silhouette_initial_clustering"] = sil_per_cell


INFO     File models/mrvi_model-2_1_32_1/model.pt already downloaded                                               


/home/geyfmand/.conda/envs/274e/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/geyfmand/.conda/envs/274e/lib/python3.11/site- ...


# DecoderZX Initialized with 2 hidden and 1 layers
# EncoderUZ Initialized with 32 hidden and 1 layers
Global silhouette score (u vs initial_clustering): 0.17640581727027893
